# Level 10: Quantum Error Correction
## Qiskit Edition

In this notebook, you will:

1. Understand why quantum errors are different from classical errors
2. Implement the **3-qubit bit-flip code**
3. Implement the **3-qubit phase-flip code**
4. See a preview of the **9-qubit Shor code** (corrects both!)
5. Understand the road to **fault-tolerant quantum computing**

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()
print("Ready!")

---
## 1. The Quantum Error Problem

### Classical vs Quantum Errors

| Classical | Quantum |
|-----------|--------|
| Bit flip: 0 → 1 | Bit flip ($X$): $|0\rangle \to |1\rangle$ |
| That's it! | Phase flip ($Z$): $|+\rangle \to |-\rangle$ |
| Can copy bits | **No-cloning theorem**: Can't copy qubits! |
| Can inspect bits | Measurement destroys superposition! |

### The Key Insight: Syndrome Measurement
Instead of measuring the data qubit directly, we use **ancilla qubits** to detect *whether an error occurred* without learning the data value.

---
## 2. The 3-Qubit Bit-Flip Code

Encodes 1 logical qubit into 3 physical qubits:

$$|0_L\rangle = |000\rangle, \quad |1_L\rangle = |111\rangle$$

If one qubit flips, majority vote corrects it!

In [ ]:
def bit_flip_encode(qc, data, ancilla1, ancilla2):
    """Encode: |ψ> → |ψψψ> using CNOTs."""
    qc.cx(data, ancilla1)
    qc.cx(data, ancilla2)

def bit_flip_detect_and_correct(qc, data, ancilla1, ancilla2, syndrome):
    """Detect and correct a single bit-flip error."""
    # Syndrome measurement: compare pairs without measuring data
    qc.cx(data, syndrome[0])
    qc.cx(ancilla1, syndrome[0])
    
    qc.cx(data, syndrome[1])
    qc.cx(ancilla2, syndrome[1])
    
    qc.measure(syndrome[0], 0)
    qc.measure(syndrome[1], 1)
    
    # Correction based on syndrome
    with qc.if_test((0, 1)):  # syndrome 01: qubit 2 flipped
        with qc.if_test((1, 0)):  # syndrome 10: not this case
            pass
    # Simplified: we'll just check syndrome pattern

# Better approach: show the full encode → error → decode → check pipeline
def bit_flip_code_demo(error_qubit=None):
    """Full 3-qubit bit-flip code demonstration."""
    data = QuantumRegister(3, 'data')
    syndrome = QuantumRegister(2, 'syndrome')
    cr_syndrome = ClassicalRegister(2, 'syn')
    cr_result = ClassicalRegister(3, 'result')
    
    qc = QuantumCircuit(data, syndrome, cr_syndrome, cr_result)
    
    # Prepare initial state |+> = H|0>
    qc.h(data[0])
    qc.barrier()
    
    # ENCODE: |ψ> → |ψψψ>
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])
    qc.barrier()
    
    # INTRODUCE ERROR (X gate on one qubit)
    if error_qubit is not None:
        qc.x(data[error_qubit])
    qc.barrier()
    
    # SYNDROME MEASUREMENT
    qc.cx(data[0], syndrome[0])
    qc.cx(data[1], syndrome[0])
    qc.cx(data[0], syndrome[1])
    qc.cx(data[2], syndrome[1])
    qc.measure(syndrome, cr_syndrome)
    qc.barrier()
    
    # CORRECTION based on syndrome
    # Syndrome 00 → no error
    # Syndrome 01 → error on qubit 2
    # Syndrome 10 → error on qubit 1
    # Syndrome 11 → error on qubit 0
    with qc.if_test((cr_syndrome, 0b01)):
        qc.x(data[2])
    with qc.if_test((cr_syndrome, 0b10)):
        qc.x(data[1])
    with qc.if_test((cr_syndrome, 0b11)):
        qc.x(data[0])
    qc.barrier()
    
    # DECODE
    qc.cx(data[0], data[2])
    qc.cx(data[0], data[1])
    qc.barrier()
    
    # Undo H to check if we recovered the original |0>
    qc.h(data[0])
    qc.measure(data, cr_result)
    
    return qc

# Show the circuit
qc = bit_flip_code_demo(error_qubit=1)
qc.draw('mpl', fold=50)

In [ ]:
# Test error correction: try error on each qubit
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scenarios = [
    (None, "No Error"),
    (0, "Error on Qubit 0"),
    (1, "Error on Qubit 1"),
    (2, "Error on Qubit 2")
]

for ax, (err, title) in zip(axes.flat, scenarios):
    qc = bit_flip_code_demo(error_qubit=err)
    result = simulator.run(qc, shots=4000).result().get_counts()
    plot_histogram(result, ax=ax, title=title)

plt.suptitle("3-Qubit Bit-Flip Code: Error Correction Results", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("If correction works: data qubit returns to |0> → result register shows '000'")
print("The syndrome register tells us WHERE the error was.")

---
## 3. The 3-Qubit Phase-Flip Code

Phase flips are the second type of quantum error: $Z|+\rangle = |-\rangle$

Trick: Work in the Hadamard (X) basis! Apply H before and after the bit-flip code.

In [ ]:
def phase_flip_code_demo(error_qubit=None):
    """Full 3-qubit phase-flip code demonstration."""
    data = QuantumRegister(3, 'data')
    syndrome = QuantumRegister(2, 'syn')
    cr_syn = ClassicalRegister(2, 'syndrome')
    cr_res = ClassicalRegister(3, 'result')
    
    qc = QuantumCircuit(data, syndrome, cr_syn, cr_res)
    
    # Initial state: |1>
    qc.x(data[0])
    qc.barrier()
    
    # ENCODE for phase-flip: switch to X-basis first
    qc.cx(data[0], data[1])
    qc.cx(data[0], data[2])
    qc.h(data[0])
    qc.h(data[1])
    qc.h(data[2])
    qc.barrier()
    
    # INTRODUCE PHASE ERROR (Z gate)
    if error_qubit is not None:
        qc.z(data[error_qubit])
    qc.barrier()
    
    # Switch back to Z-basis for syndrome measurement
    qc.h(data[0])
    qc.h(data[1])
    qc.h(data[2])
    
    # SYNDROME MEASUREMENT (same as bit-flip code now)
    qc.cx(data[0], syndrome[0])
    qc.cx(data[1], syndrome[0])
    qc.cx(data[0], syndrome[1])
    qc.cx(data[2], syndrome[1])
    qc.measure(syndrome, cr_syn)
    qc.barrier()
    
    # CORRECTION
    with qc.if_test((cr_syn, 0b01)):
        qc.x(data[2])
    with qc.if_test((cr_syn, 0b10)):
        qc.x(data[1])
    with qc.if_test((cr_syn, 0b11)):
        qc.x(data[0])
    qc.barrier()
    
    # DECODE
    qc.cx(data[0], data[2])
    qc.cx(data[0], data[1])
    qc.measure(data, cr_res)
    
    return qc

# Test phase-flip correction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (err, title) in zip(axes, [(None, "No Phase Error"), (1, "Phase Error on Qubit 1")]):
    qc = phase_flip_code_demo(error_qubit=err)
    result = simulator.run(qc, shots=4000).result().get_counts()
    plot_histogram(result, ax=ax, title=title)

plt.suptitle("3-Qubit Phase-Flip Code", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Result should show '001' (original |1>) even after phase error!")

---
## 4. Noisy Channel Correction

Let's use a probabilistic noise model to show error correction in action.

In [ ]:
def unprotected_qubit(p_error):
    """Single qubit through a bit-flip noise channel."""
    qc = QuantumCircuit(1, 1)
    qc.x(0)  # Prepare |1>
    qc.id(0)  # Identity — noise will be applied here
    qc.measure(0, 0)
    
    # Noise model: bit-flip with probability p
    noise_model = NoiseModel()
    error = pauli_error([('X', p_error), ('I', 1 - p_error)])
    noise_model.add_all_qubit_quantum_error(error, ['id'])
    
    noisy_sim = AerSimulator(noise_model=noise_model)
    counts = noisy_sim.run(qc, shots=10000).result().get_counts()
    correct = counts.get('1', 0) / 10000
    return correct

def protected_qubit_3bit(p_error):
    """3-qubit bit-flip code through a noisy channel."""
    qc = QuantumCircuit(5, 5)  # 3 data + 2 syndrome
    
    # Prepare |1>
    qc.x(0)
    
    # Encode
    qc.cx(0, 1)
    qc.cx(0, 2)
    qc.barrier()
    
    # Noise channel (id gate on each data qubit)
    qc.id(0)
    qc.id(1)
    qc.id(2)
    qc.barrier()
    
    # Syndrome
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.cx(0, 4)
    qc.cx(2, 4)
    qc.measure([3, 4], [0, 1])
    
    # Correction
    cr = qc.cregs[0]
    with qc.if_test((cr, 0b01)):
        qc.x(2)
    with qc.if_test((cr, 0b10)):
        qc.x(1)
    with qc.if_test((cr, 0b11)):
        qc.x(0)
    
    # Decode
    qc.cx(0, 2)
    qc.cx(0, 1)
    qc.measure(0, 2)
    
    # Noise model
    noise_model = NoiseModel()
    error = pauli_error([('X', p_error), ('I', 1 - p_error)])
    noise_model.add_all_qubit_quantum_error(error, ['id'])
    
    noisy_sim = AerSimulator(noise_model=noise_model)
    counts = noisy_sim.run(qc, shots=10000).result().get_counts()
    
    # Check if data qubit 0 is correct (should be |1>)
    correct = 0
    for bitstring, count in counts.items():
        # Bit at index 2 corresponds to data qubit 0
        if bitstring[-3] == '1':  # Third classical bit
            correct += count
    return correct / 10000

# Compare unprotected vs protected over range of error rates
error_rates = np.linspace(0.001, 0.3, 20)
unprotected = [unprotected_qubit(p) for p in error_rates]
protected = [protected_qubit_3bit(p) for p in error_rates]

plt.figure(figsize=(10, 6))
plt.plot(error_rates, unprotected, 'r-o', label='Unprotected (1 qubit)')
plt.plot(error_rates, protected, 'b-s', label='3-Qubit Bit-Flip Code')
plt.xlabel('Physical Error Rate (p)')
plt.ylabel('Probability of Correct Result')
plt.title('Error Correction: Unprotected vs 3-Qubit Code')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("The 3-qubit code wins when p is small (< ~0.15).")
print("At higher error rates, the code actually performs WORSE!")
print("This is the 'threshold' concept — error correction only helps below a threshold.")

---
## 5. The Road Ahead

### Error Correction Codes

| Code | Corrects | Physical Qubits per Logical |
|------|----------|----------------------------|
| 3-qubit bit-flip | 1 bit-flip | 3 |
| 3-qubit phase-flip | 1 phase-flip | 3 |
| **Shor [[9,1,3]]** | 1 arbitrary error | 9 |
| **Steane [[7,1,3]]** | 1 arbitrary error | 7 |
| **Surface Code** | Many errors | ~1000 per logical |

### The Surface Code
- Most promising for real hardware
- Qubits arranged on a 2D grid
- Threshold error rate: ~1%
- Current hardware is **approaching** this threshold!

### The Big Picture
```
         ┌─────────────────────────────────────────────────┐
         │             Fault-Tolerant QC                   │
         │  Millions of physical qubits → thousands of     │
         │  logical qubits → Shor's on RSA-2048            │
         ├─────────────────────────────────────────────────┤
         │             We are here: NISQ Era               │
         │  ~1000 physical qubits, no error correction     │
         │  VQE, QAOA, small demonstrations                │
         └─────────────────────────────────────────────────┘
```

---
## 6. Key Takeaways

| Concept | Description |
|---------|-------------|
| **Bit-flip code** | 3 qubits, majority vote in Z-basis |
| **Phase-flip code** | 3 qubits, majority vote in X-basis |
| **Syndrome** | Ancilla measurement reveals error location |
| **Threshold** | Error correction helps only below a threshold rate |
| **Surface code** | Most practical path to fault-tolerant QC |

---
## 7. Final Challenges

1. **Various initial states**: Test the bit-flip code with $|+\rangle$ and arbitrary rotation states.
2. **Two errors**: What happens if 2 out of 3 qubits get flipped? Can the code still correct?
3. **Shor code**: Implement the 9-qubit Shor code that combines bit-flip and phase-flip protection.

### Congratulations!
You've completed the Introduction to Quantum Computing course! You now understand:
- Qubits, gates, and measurement (Levels 1-2)
- Entanglement and protocols (Levels 3-4)
- Quantum algorithms (Levels 5-8)
- NISQ-era practical computing (Level 9)
- Quantum error correction (Level 10)

In [ ]:
# Your challenge code here!

---
**Back to: [Course README](../README.md)**